# Plot Coverage Plots for Multiome data

### Load packages

In [ ]:
library(future)
library(Seurat)
library(Signac)
library(GenomicRanges)
library(scales)
library(tidyverse)
library(ggplot2)
library(dplyr)

source("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/global.R")

# Set random seed for reproducibility
set.seed(1)

# Memory and compatibility options
options(
  future.globals.maxSize = 3e+09,
  Seurat.object.assay.version = "v3"
)

# Set figure directory
figdir <- "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/Multiome"

### Load data

In [ ]:
# Load integrated annotations
multiome_integrated_annotations <- read.csv('/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/10X.multiome.processed.controls.integrated.cell_metadata.csv')

In [ ]:
multiome_integrated_annotations %>% head()

In [ ]:
data.seurat.multiome <- readRDS(paste0(io$outdir.processed, "/parse.multiome.annotated.rds"))

In [ ]:
data.seurat.multiome.list <- SplitObject(data.seurat.multiome, split.by="batch")

In [ ]:
data.seurat.multiome.list$Multiome1@assays$ATAC@fragments[[1]]

In [ ]:
clean_seurat <- function(obj, assays_to_remove = c("peaks", "SCT"),
                         reductions_to_remove = c("pca", "umap.pca", "lsi", "umap.lsi")) {
  # remove assays
  for (assay in assays_to_remove) {
    if (assay %in% Assays(obj)) {
      obj[[assay]] <- NULL
    }
  }
  # remove reductions
  for (red in reductions_to_remove) {
    if (red %in% Reductions(obj)) {
      obj@reductions[[red]] <- NULL
    }
  }
  return(obj)
}

In [ ]:
# Apply to each element in the list
data.seurat.multiome.list <- lapply(
  data.seurat.multiome.list,
  clean_seurat
)

In [ ]:
colnames(data.seurat.multiome.list[[1]]) <- paste0(colnames(data.seurat.multiome.list[[1]]), "-ITBOGE013_YBP2_14_Multiome_1")
colnames(data.seurat.multiome.list[[2]]) <- paste0(colnames(data.seurat.multiome.list[[2]]), "-ITBOGE013_YBP2_14_Multiome_2")

In [ ]:
data.seurat.multiome.merged <- merge(
  x = data.seurat.multiome.list[[1]],
  y = data.seurat.multiome.list[[2]]
)

In [ ]:
data.seurat.multiome3 <- readRDS(paste0(io$outdir.processed, "/10X.multiome.processed._20250227_190214.annotated.rds"))

In [ ]:
data.seurat.multiome3 <- subset(data.seurat.multiome3, subset = condition %in% c("Control"))

In [ ]:
colnames(data.seurat.multiome3) <- paste0(colnames(data.seurat.multiome3), "-ITBOGE001_Fujii")

In [ ]:
data.seurat.multiome3 <- clean_seurat(obj = data.seurat.multiome3)

In [ ]:
data.seurat.multiome.merged[["RNA"]] <- as(data.seurat.multiome.merged[["RNA"]], "Assay")
data.seurat.multiome3[["RNA"]] <- as(data.seurat.multiome3[["RNA"]], "Assay")

In [ ]:
Key(data.seurat.multiome.merged[["RNA"]])  <- "RNA_"
Key(data.seurat.multiome.merged[["ATAC"]]) <- "ATAC_"

Key(data.seurat.multiome3[["RNA"]])  <- "RNA_"
Key(data.seurat.multiome3[["ATAC"]]) <- "ATAC_"

In [ ]:
Assays(data.seurat.multiome.merged)
Assays(data.seurat.multiome3)

# Keys prüfen
lapply(Assays(data.seurat.multiome.merged), function(a) Key(data.seurat.multiome.merged[[a]]))
lapply(Assays(data.seurat.multiome3), function(a) Key(data.seurat.multiome3[[a]]))


In [ ]:
data.seurat.multiome.merged <- merge(
  x = data.seurat.multiome.merged,
  y = data.seurat.multiome3
)

In [ ]:
# Get fragments from individual objects
fragments1 <- data.seurat.multiome.list[[1]]@assays$ATAC@fragments[[1]]
fragments2 <- data.seurat.multiome.list[[2]]@assays$ATAC@fragments[[2]]
fragments3 <- data.seurat.multiome3@assays$ATAC@fragments[[3]]

In [ ]:
names(fragments1@cells) <- paste0(names(fragments1@cells), "-ITBOGE013_YBP2_14_Multiome_1")
names(fragments2@cells) <- paste0(names(fragments2@cells), "-ITBOGE013_YBP2_14_Multiome_2")
names(fragments3@cells) <- paste0(names(fragments3@cells), "-ITBOGE001_Fujii")

In [ ]:
frags <- list(fragments1, fragments2, fragments3)

In [ ]:
Fragments(data.seurat.multiome.merged[["ATAC"]]) <- frags

In [ ]:
multiome_integrated_annotations <- multiome_integrated_annotations %>% column_to_rownames("index")

In [ ]:
data.seurat.multiome.merged <- subset(data.seurat.multiome.merged, cells=rownames(multiome_integrated_annotations))

In [ ]:
data.seurat.multiome.merged@meta.data <- multiome_integrated_annotations

In [ ]:
DefaultAssay(data.seurat.multiome.merged) <- "ATAC"

In [ ]:
saveRDS(data.seurat.multiome.merged, paste0(io$outdir.processed, "/parse.multiome.merged.cont3.rds"))

## Create stacked barplot of cell type composition per sample

In [ ]:
data.seurat.multiome.merged <- readRDS(paste0(io$outdir.processed, "/parse.multiome.merged.cont3.rds"))

In [ ]:
data.seurat.multiome.merged@assays$ATAC@fragments[[10]]@path

In [ ]:
ct_colors <- c(
  # Proliferative axis
  "Stem Cells"       = "#1f77b4",  # deep blue
  "TA Cells"         = "#4c91c6",  # medium blue
  "Cycling Cells"    = "#7aaed6",  # light blue
  
  # Absorptive lineage
  "Enterocytes"      = "#2ca02c",  # green
  "BEST4+ Enterocytes" = "#1abc9c",# teal
  
  # Secretory lineage
  "Goblet Cells"     = "#ff7f0e",  # orange
  
  # EEC progenitors
  "EEC Progenitors"  = "#9467bd",  # purple
  
  # EEC subtypes
  "EC Cells"         = "#d62728",  # red
  "D Cells"          = "#17becf",  # cyan
  "X Cells"          = "#e377c2",  # magenta
  "I/N Cells"        = "#7f7f7f",  # grey
  "K Cells"          = "#bcbd22"   # olive
)

In [ ]:
data.seurat.multiome.merged

In [ ]:
# set factor levels according to ct_colors names
data.seurat.multiome.merged$final_annotation <- factor(
  data.seurat.multiome.merged$final_annotation,
  levels = names(ct_colors)  # enforce order
)

In [ ]:
# Extract the data from your Seurat object
plot_data <- data.seurat.multiome.merged@meta.data %>%
  as.data.frame() %>%
  rownames_to_column("cell_id") %>%
  select(cell_id, sample, final_annotation)

# Create the bar plot
p <- ggplot(plot_data, aes(x = sample, fill = final_annotation)) +
  geom_bar(position = "fill") +
  coord_flip() +
  scale_fill_manual(values = ct_colors) +
  labs(
    x = "Sample",
    y = "Proportion",
    fill = "Cell Type",
    title = "Cell Type Distribution by Sample"
  ) +
  theme_minimal() +
  theme(
    axis.text = element_text(size = 10),
    legend.position = "right"
  )

ggsave(filename = paste0(figdir, "/cell_type_distribution_by_sample.pdf"), plot = p, width = 8, height = 6, dpi = 300)

## Plot coverage for selected markers

In [ ]:
p <- CoveragePlot(
  object = data.seurat.multiome.merged,
  region = "GHRL",         # gene name works if Annotation() is set
  annotation = TRUE,       # adds gene models from the annotation
  peaks = FALSE,            # show called peaks
  extend.upstream = 10000,
  extend.downstream = 10000,
  group.by = "final_annotation"   # split tracks by a metadata column
)

p <- p & scale_fill_manual(values=ct_colors)

ggsave(
  filename = paste0(figdir, "/CoveragePlot_GHRL_multiome_merged_cont3.png"),
  plot = p,
  width = 12,
  height = 10,
  dpi = 300
)

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 10)

genes <- c("GHRL", "NEUROG3", "CHGA", "CHGB", "TAC1", "TPH1", "GCG", "PYY", "SST", "NTS", "CCK", "GIP", "MLN")
#genes <- c("RFX6", "PLAGL1")

for(gene in genes) {
  p <- CoveragePlot(
    object = data.seurat.multiome.merged,
    region = gene,         # gene name works if Annotation() is set
    annotation = TRUE,       # adds gene models from the annotation
    peaks = FALSE,            # show called peaks
    extend.upstream = 10000,
    extend.downstream = 10000,
    window=500,
    group.by = "final_annotation"   # split tracks by a metadata column
  )

  p <- p & scale_fill_manual(values=ct_colors)

  ggsave(paste0(figdir, "/multiome_merged_coverage_plot_", gene, ".pdf"), p, width=6, height=5)

}

## Plot coverage for variant associations

In [ ]:
gwas.dir <- "/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/external/data/gwas"
gwas.files <- list.files(gwas.dir, pattern = "*.tsv", full.names = TRUE)
gwas.df <- do.call(rbind, lapply(gwas.files, read.csv, sep="\t"))

gwas.granges <- gwas.df %>% 
    separate(locations, c("chr", "pos"), sep=":") %>%
    mutate(chr = paste0("chr", chr),
           start = as.numeric(pos),
           end = as.numeric(pos)+1) %>%
    dplyr::select(chr, start, end, riskAllele) %>% 
    makeGRangesFromDataFrame(keep.extra.columns=TRUE)

In [ ]:
seu_obj <- data.seurat.multiome.merged
annotation <- Annotation(seu_obj)
names(annotation) <- NULL

extend.upstream <- 10000
extend.downstream <- 10000

In [ ]:
genes <- c("GHRL", "NTS", "PYY", "CCK", "GIP", "MLN")

for(gene in genes) {
  gene_annotation <- annotation %>% as.data.frame() %>% dplyr::filter(gene_name == gene)
  gene_start <- min(gene_annotation$start)
  gene_end <- max(gene_annotation$end)
  gene_chr <- unique(gene_annotation$seqnames)
  region <- GRanges(seqnames=gene_chr, ranges=IRanges(start=gene_start-extend.downstream, end=gene_end+extend.upstream))

  gwas.granges.sub <- subsetByOverlaps(gwas.granges, region)
  snp.df <- as.data.frame(x = gwas.granges.sub)

  p_list <- list()
  p_list[["coverage"]] <- CoveragePlot(
      object = seu_obj,
      region = gene,         # gene name works if Annotation() is set
      annotation = FALSE,       # adds gene models from the annotation
      peaks = FALSE,            # show called peaks
      extend.upstream = extend.upstream,
      extend.downstream = extend.downstream,
      window=500,
      group.by = "final_annotation"   # split tracks by a metadata column
  )
  p_list[["coverage"]] <- p_list[["coverage"]] & scale_fill_manual(values=ct_colors)
  p_list[["SNP"]] <- ggplot() + 
                  geom_segment(aes(x = start, y = 0, xend = end, yend = 1),
                              size =  0.5,
                              data = snp.df) + 
                  geom_text(aes(x = start, y = 1, label = riskAllele),
                            size = 3,
                            data = snp.df,
                            max.overlaps=Inf) +
                  theme_void()
                  
  gene_plot <- AnnotationPlot(
                object = seu_obj,
                region = region,
                extend.upstream = extend.upstream,
                extend.downstream = extend.downstream
              )
  p_list[["gene"]] <- gene_plot

  p <- CombineTracks(
          plotlist = p_list,
          )
 
  ggsave(paste0(figdir, "/multiome_merged_coverage_plot_", gene, "_with_GWAS_snps.pdf"), p, width=10, height=9)

}

### Plot for milestones

In [ ]:
library(stringr)

# Function to parse barcodes
parse_bc <- function(x) {
  # Pattern A: <cell>-1_<lane>-<sample>
  pat_A <- "^(.+?-1)_[0-9]+-(.+)$"
  
  # Pattern B: <cell>-1-<sample>
  pat_B <- "^(.+?-1)-(.+)$"
  
  out <- character(length(x))
  
  # Apply pattern A
  idx_A <- str_detect(x, pat_A)
  out[idx_A] <- str_replace(x[idx_A], pat_A, "\\2#\\1")
  
  # Apply pattern B
  idx_B <- str_detect(x, pat_B)
  out[idx_B] <- str_replace(x[idx_B], pat_B, "\\2#\\1")
  
  return(out)
}

In [ ]:
ct_colors = c(
    "NGN3+ cells" =        "#9467bd",  # purple
    "EC/X/D/I/K cells" =  "#a871b0",  # purple → red transition
    "EC cells" =          "#c75a6e",  # muted warm red
    "X/D/I/K cells" =     "#d89e40",  # olive/gold
    "X cells" =           "#e377c2",  # magenta
    "D/I/K cells" =       "#c7a7d2",  # ★ new soft lavender-rose (no grey)
    "D cells" =           "#17becf",  # cyan
    "I/K cells" =         "#bcbd22"  # olive-green
)

In [ ]:
# Load merged multiome data
data.seurat.multiome.merged <- readRDS(file = paste0(io$outdir.processed, "/parse.multiome.merged.cont3.eecs_for_pando.rds"))
data.seurat.multiome.merged@meta.data$formatted_barcodes <- parse_bc(rownames(data.seurat.multiome.merged@meta.data))

# Load milestone annotations
milestone.annotations <- read.csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/grn/multiome_chromvar_mapped_vasa_milestones_annotations.csv", header=TRUE, row.names = 1)
milestone.annotations <- milestone.annotations %>% rownames_to_column("formatted_barcodes")

metadata <- data.seurat.multiome.merged@meta.data %>% rownames_to_column(var="index")
metadata <- metadata %>% left_join(milestone.annotations, by=c("formatted_barcodes" = "formatted_barcodes"))
data.seurat.multiome.merged@meta.data <- metadata %>% column_to_rownames("index")

# Fill NA milestones with "non-eec"
data.seurat.multiome.merged@meta.data$milestones <- data.seurat.multiome.merged@meta.data$milestones %>% replace_na("non-eec")

# Subset to keep only cells that are not "non-eec"
keep.cells <- data.seurat.multiome.merged@meta.data %>% 
                dplyr::filter(milestones != "non-eec") %>%
                rownames()
data.seurat.multiome.merged <- subset(data.seurat.multiome.merged, cells=keep.cells)

DefaultAssay(data.seurat.multiome.merged) <- "ATAC"

# set factor levels according to ct_colors names
data.seurat.multiome.merged$milestones <- factor(
  data.seurat.multiome.merged$milestones,
  levels = names(ct_colors)  # enforce order
)


In [ ]:
# Check regions 
regions <- c("chr7-127809012-127809923", "chr7-128123281-128124222", "chr2-218126880-218127782", "chr8-72864157-72865152")

for (region in regions) {
  p <- CoveragePlot(
    object = data.seurat.multiome.merged,
    region = region,         
    annotation = TRUE,      
    peaks = TRUE,          
    window = 500,
    region.highlight = StringToGRanges(region),
    extend.upstream = 10000,
    extend.downstream = 10000,
    group.by = "milestones"   # split tracks by a metadata column
  ) 
  p <- p & scale_fill_manual(values=ct_colors)
  ggsave(
    filename = paste0(figdir, "/CoveragePlot_region_", region, "_multiome_merged_eec_only.pdf"),
    plot = p,
    width = 5,
    height = 3,
    dpi = 300
  )
}

In [ ]:
paste0(figdir, "/CoveragePlot_region_", region, "_multiome_merged_eec_only.pdf")